# Function expansion

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

## Tagging

In [ ]:
from typing import List,Optional
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

#   Evaluate user's prompt
class Tagging(BaseModel):
    """Tag the piece of text with particular info."""
    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, or `neutral`")
    language: str = Field(description="language of text (should be ISO 639-1 code)")

tagging_functions = [convert_to_openai_function(Tagging)]
tagging_functions

c:\Users\60410\Desktop\claude code\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[{'name': 'Tagging',
  'description': 'Tag the piece of text with particular info.',
  'parameters': {'properties': {'sentiment': {'description': 'sentiment of text, should be `pos`, `neg`, or `neutral`',
     'type': 'string'},
    'language': {'description': 'language of text (should be ISO 639-1 code)',
     'type': 'string'}},
   'required': ['sentiment', 'language'],
   'type': 'object'}}]

In [16]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

model = ChatOpenAI(model="deepseek-chat", temperature=0,
                  api_key=os.getenv("DEEPSEEK_API_KEY"),
                  base_url=os.getenv("DEEPSEEK_BASE_URL"))

model_with_functions = model.bind_tools(
    tagging_functions,
    tool_choice="Tagging"
)

tagging_chain = prompt | model_with_functions

response=tagging_chain.invoke({"input": "I love shanghai"})
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 350, 'total_tokens': 403, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 94}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '214caa1b-fa68-46dc-a283-efb1554ffd09', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f4233-5443-7903-b446-306d8cf58199-0', tool_calls=[{'name': 'Tagging', 'args': {'sentiment': 'pos', 'language': 'en'}, 'id': 'call_00_Qv7QD7d5W7UkKVcFEKom4014', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 350, 'output_tokens': 53, 'total_tokens': 403, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}})

In [6]:
arguments=response.tool_calls[0]['args']

arguments['sentiment']

'pos'

## Extraction

In [7]:
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="person's name")
    age: Optional[int] = Field(description="person's age")
        
class Information(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="List of info about people")

In [13]:
extraction_functions = [convert_to_openai_function(Information)]

extraction_functions

[{'name': 'Information',
  'description': 'Information to extract.',
  'parameters': {'properties': {'people': {'description': 'List of info about people',
     'items': {'description': 'Information about a person.',
      'properties': {'name': {'description': "person's name",
        'type': 'string'},
       'age': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
        'description': "person's age"}},
      'required': ['name', 'age'],
      'type': 'object'},
     'type': 'array'}},
   'required': ['people'],
   'type': 'object'}}]

In [15]:
from langchain_core.output_parsers.openai_tools import PydanticToolsParser
 
#   In the absence of explicitly provided information, do not guess in order to avoid hullucination
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")
])

extraction_model = model.bind_tools(extraction_functions, 
                              tool_choice="Information")

extraction_chain = prompt2 | extraction_model | PydanticToolsParser(tools=[Information])
 

content2=extraction_chain.invoke({"input": "Henry is 17 years old, and he has a friend Michael. "})
content2

[Information(people=[Person(name='Henry', age=17), Person(name='Michael', age=None)])]

In [17]:
type(content2)

list

# Application
Summary and extract content from websites.

In [24]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.channelnewsasia.com/singapore/temasek-ai-portfolio-investments-private-credit-infrastructure-6240171")
documents = loader.load()

doc = documents[0]
page_content = doc.page_content[:3000]

In [23]:
class Overview(BaseModel):
    """Overview of a section of text."""
    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")

class News(BaseModel):
    """Information about news mentioned."""
    title: str
    author: Optional[str]
 
 
class Info(BaseModel):
    """Information to extract"""
    news: List[News]

overview_tagging_function = [
    convert_to_openai_function(Overview), convert_to_openai_function(Tagging), 
    convert_to_openai_function(News),convert_to_openai_function(Info)
]

tagging_model = model.bind_tools(
    overview_tagging_function,
    tool_choice="auto"  # Default;"any": must choose one;"none": no tools used
)

prompt3 = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")
])

tagging_chain = prompt3 | tagging_model | PydanticToolsParser(tools=[Overview, Tagging, News, Info])    # Must include all the tool functions

tagging_chain.invoke({"input": page_content})

[Overview(summary='Temasek, a Singapore investment firm, aims to more than double its AI investment portfolio share to as much as 15% by 2031.', language='English', keywords='Temasek, AI, investment, portfolio, 2031'),
 Tagging(sentiment='neutral', language='en'),
 Info(news=[News(title='Temasek aims to more than double AI investment portfolio share to as much as 15% by 2031', author=None)])]

In [26]:
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="person's name")
    Company: Optional[str] = Field(description="person's company")
        
class PersonInfo(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="List of info about people")

news_extraction_function = [
    convert_to_openai_function(PersonInfo)
]

extraction_model = model.bind_tools(
    news_extraction_function, 
    tool_choice="PersonInfo"
)

prompt4 = ChatPromptTemplate.from_messages([
    ("system", "A article will be passed to you. Extract from it all people  that are mentioned by this article. If no people are mentioned that's fine - you don't need to extract any! Just return an empty list.Do not make up or guess ANY extra information. Only extract what exactly is in the text. "),
    ("human", "{input}")
])

extraction_chain = prompt4 | extraction_model | PydanticToolsParser(tools=[PersonInfo])  

extraction_chain.invoke({"input": page_content})


[PersonInfo(people=[])]

## Text split

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableLambda     #   Store chunked text in a list

text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)
 
prep = RunnableLambda(
    lambda x: [{"input": doc} for doc in text_splitter.split_text(x)]
)
 
response = prep.invoke(doc.page_content)

In [28]:
len(response)

8

In [ ]:
def flatten_results(nested_list):   # flatten matrix and remove empty item
    results = []
    for sublist in nested_list:
        for info in sublist:
            for person in info.people:
                results.append(person)
    return results

flatten = RunnableLambda(flatten_results)

In [35]:
chain = prep | extraction_chain.map() | flatten     #   Batching
chain.invoke(doc.page_content)

[Person(name='Dilhan Pillay', Company='Temasek Holdings'),
 Person(name='Mr Pillay', Company='Temasek'),
 Person(name='Mr Chia Song Hwee', Company='Temasek Global Investments'),
 Person(name='Mr Nagi Hamiyeh', Company='Temasek Global Investments'),
 Person(name='Png Chin Yee', Company='Temasek'),
 Person(name='Mr Pillay', Company='Temasek'),
 Person(name='Mr Chia', Company=None),
 Person(name='Chia Der Jiun', Company='Monetary Authority of Singapore'),
 Person(name='Chia Song Hwee', Company='Temasek'),
 Person(name='Mr Chia', Company='Temasek'),
 Person(name='Rohit Sipahimalani', Company='Temasek International'),
 Person(name='Gabriel Lim', Company='Seviora Holdings')]